## Example 1: Arrays: from Python lists and pyarrow, and back

In [ ]:
import importlib.metadata

import pyarrow as pa
from arro3.core import Array, DataType

# Example 1: an arro3 Array shares Arrow memory with pyarrow through the Arrow C data interface
assert importlib.metadata.version("arro3-core") == "0.9.1"
arr = Array([1, 2, None, 4], type=DataType.int64())
assert arr.null_count == 1 and arr.to_pylist() == [1, 2, None, 4]
pa_arr = pa.array(arr)
assert pa_arr.equals(pa.array([1, 2, None, 4], pa.int64()))
back = Array.from_arrow(pa.array(["a", "b"]))
assert back.type.equals(DataType.string()) and back.to_pylist() == ["a", "b"]
print("Example 1 OK:", arr.type, len(pa_arr))

## Example 2: Tables: from a dict, columns, slices and pyarrow interchange

In [ ]:
import pyarrow as pa
from arro3.core import Array, DataType, Table

# Example 2: a Table built from a dict of Arrow columns, selected, sliced, and handed to pyarrow
ids = Array([1, 2, 3, 4], type=DataType.int64())
t = Table.from_pydict({"id": ids, "name": Array(["a", "b", "c", "d"], type=DataType.string())})
assert t.num_rows == 4 and t.column_names == ["id", "name"]
sub = t.select(["name"]).slice(1, 2)
assert sub.shape == (2, 1)
pt = pa.table(sub)
assert pt.column("name").to_pylist() == ["b", "c"], pt
assert Table.from_arrow(pa.table({"x": [9]})).num_rows == 1
print("Example 2 OK:", t.shape, pt.to_pydict())

## Example 3: NumPy round trip

In [ ]:
import numpy as np
from arro3.core import Array, DataType

# Example 3: numeric arrays move to and from NumPy without changing values
src = np.arange(10, dtype=np.float64) / 4
arr = Array.from_numpy(src)
assert arr.type.equals(DataType.float64()) and len(arr) == 10
out = arr.to_numpy()
assert np.array_equal(out, src) and out.dtype == np.float64
assert arr.take(Array([0, 9], type=DataType.int32())).to_pylist() == [0.0, 2.25]
print("Example 3 OK:", out[:4])

## Example 4: Schemas, fields and casting

In [ ]:
from arro3.core import Array, DataType, Field, Schema

# Example 4: a schema of named, typed fields; an int32 array cast to int64 and to string
schema = Schema([Field("x", DataType.int32(), nullable=False), Field("y", DataType.string())])
assert schema.names == ["x", "y"] and schema.field("x").nullable is False
a32 = Array([1, 2, 3], type=DataType.int32())
assert a32.cast(DataType.int64()).type.equals(DataType.int64())
assert a32.cast(DataType.string()).to_pylist() == ["1", "2", "3"]
print("Example 4 OK:", schema.names)

## Example 5: Record batches, chunked arrays, and the error for unequal columns

In [ ]:
from arro3.core import Array, ChunkedArray, DataType, RecordBatch, Table

# Example 5: a table made of two record batches has two chunks; columns of unequal length are refused
b1 = RecordBatch.from_pydict({"v": Array([1, 2], type=DataType.int64())})
b2 = RecordBatch.from_pydict({"v": Array([3], type=DataType.int64())})
t = Table.from_batches([b1, b2])
assert t.num_rows == 3 and t.chunk_lengths == [2, 1]
col = t.column("v")
assert isinstance(col, ChunkedArray) and col.combine_chunks().to_pylist() == [1, 2, 3]
try:
    Table.from_pydict({"a": Array([1, 2], type=DataType.int64()), "b": Array([1], type=DataType.int64())})
    raise AssertionError("expected an error for unequal column lengths")
except ValueError as error:
    assert "same chunk lengths" in str(error), error
    print("Example 5 OK: chunks", t.chunk_lengths, "| expected error:", error)